# Vortex Fitting and Detection Notebook

This notebook runs the vortex fitting script on GLORYS data using the 'thesis' kernel.

In [88]:
import sys
import os
sys.path.append('SOFTX-D-20-00015-master')

import argparse
from vortexfitting import fitting
from vortexfitting import schemes
from vortexfitting import detection
from vortexfitting import output
from vortexfitting import classes
import xarray as xr

In [89]:
import matplotlib
matplotlib.use('Agg')  # For headless plotting

In [90]:
# Set Parameters
input_filename = 'SOFTX-D-20-00015-master/data/GPGP_oct2020_22-27N_145-140W2.nc'
output_directory = 'results'
scheme = 22
detection_method = 'swirling'
detection_threshold = 0.0
box_size = 6
flip_axis = False
mean_filename = '/'
plot_method = 'fit'
xy_location = [0, 0]
first = 0
last = 0
step = 1
rmax = 0 # 8000
file_type = 'dns'
correlation_threshold = 0.6
output_format = 'png'

In [91]:
ds_test = xr.open_dataset(input_filename)
print(ds_test)

<xarray.Dataset> Size: 46kB
Dimensions:     (time: 1, depth: 1, latitude: 61, longitude: 61)
Coordinates:
  * time        (time) int64 8B 0
  * depth       (depth) float64 8B 0.0
  * latitude    (latitude) float64 488B 0.0 9.266e+03 ... 5.467e+05 5.56e+05
  * longitude   (longitude) float64 488B 0.0 8.398e+03 ... 4.955e+05 5.039e+05
Data variables:
    velocity_x  (time, depth, latitude, longitude) float32 15kB ...
    velocity_y  (time, depth, latitude, longitude) float32 15kB ...
    velocity_z  (time, depth, latitude, longitude) float32 15kB ...
Attributes:
    Conventions:       CF-1.11
    title:             daily mean fields from Global Ocean Physics Analysis a...
    institution:       MERCATOR OCEAN
    source:            MERCATOR GLORYS12V1
    history:           2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    references:        http://www.mercator-ocean.fr
    comment:           CMEMS product
    subset:source:     ARCO data downloaded from the Marine Data Store using 

In [92]:
# Load Data
if last < first:
    last = first

for time_step in range(first, last + 1, step):
    if not os.path.exists(input_filename.format(time_step)):
        print('The input file does not exist. Exiting.')
        sys.exit()

    print('\nOpening file: ', input_filename.format(time_step), ', file type: ', file_type)
    if mean_filename != '/':
        print('Opening mean field: ', mean_filename)

    vfield = classes.VelocityField(input_filename, time_step, mean_filename, file_type)


Opening file:  SOFTX-D-20-00015-master/data/GPGP_oct2020_22-27N_145-140W2.nc , file type:  dns


In [93]:
# Compute Derivatives
if scheme == 4:
    vfield.derivative = schemes.fourth_order_diff(vfield)
elif scheme == 2:
    vfield.derivative = schemes.second_order_diff(vfield)
elif scheme == 22:
    vfield.derivative = schemes.least_square_diff(vfield)
else:
    print('No scheme', scheme, 'found. Exiting!')
    sys.exit()

Difference scheme: least-square filter


In [94]:
# Compute Vorticity
vorticity = vfield.derivative['dvdx'] - vfield.derivative['dudy']

In [95]:
# Detect Vortices
detection_field = []
if detection_method == 'Q':
    detection_field = detection.calc_q_criterion(vfield)
elif detection_method == 'swirling':
    detection_field = detection.calc_swirling(vfield)
elif detection_method == 'delta':
    detection_field = detection.calc_delta_criterion(vfield)

if vfield.normalization_flag:
    print('Normalization for ', vfield.normalization_direction, ' direction')
    detection_field = fitting.normalize(detection_field, vfield.normalization_direction)

Detection method: 2D swirling strength
Max value of swirling:  0.0


In [96]:
# Find Peaks
print('Threshold=', detection_threshold, ', box size=', box_size)
peaks = fitting.find_peaks(detection_field, detection_threshold, box_size)
print('Vortices found: ', len(peaks[0]))

Threshold= 0.0 , box size= 6
Vortices found:  40


In [97]:
# Determine Rotation Direction
vortices_counterclockwise, vortices_clockwise = fitting.direction_rotation(vorticity, peaks)

In [98]:
# Fit Vortices
vortices = list()
if (plot_method == 'fit') and (xy_location == [0, 0]):
    vortices = fitting.get_vortices(vfield, peaks, vorticity, rmax, correlation_threshold)
    print('---- Accepted vortices ----')
    print(len(vortices))
else:
    print('No fitting')

0 Processing detected swirling at (x, y) 23 2
Accepted! Correlation = 0.83 (vortex # 0)
1 Processing detected swirling at (x, y) 57 2
Accepted! Correlation = 0.91 (vortex # 1)
2 Processing detected swirling at (x, y) 45 3


3 Processing detected swirling at (x, y) 38 4
4 Processing detected swirling at (x, y) 13 6
Accepted! Correlation = 0.90 (vortex # 2)
5 Processing detected swirling at (x, y) 33 12
Accepted! Correlation = 0.68 (vortex # 3)
6 Processing detected swirling at (x, y) 13 13
Accepted! Correlation = 0.91 (vortex # 4)
7 Processing detected swirling at (x, y) 43 13
8 Processing detected swirling at (x, y) 53 14
Accepted! Correlation = 0.77 (vortex # 5)
9 Processing detected swirling at (x, y) 14 20
10 Processing detected swirling at (x, y) 26 20
Accepted! Correlation = 0.66 (vortex # 6)
11 Processing detected swirling at (x, y) 2 21
Accepted! Correlation = 0.82 (vortex # 7)
12 Processing detected swirling at (x, y) 36 21
13 Processing detected swirling at (x, y) 10 23
14 Processing detected swirling at (x, y) 48 23
15 Processing detected swirling at (x, y) 6 28
16 Processing detected swirling at (x, y) 35 28
Accepted! Correlation = 0.67 (vortex # 8)
17 Processing detected swirling at (x, y) 15 

c:\Users\Jelle Gortemaker\miniconda3\envs\thesis\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,


Accepted! Correlation = 0.82 (vortex #21)
34 Processing detected swirling at (x, y) 15 54
Accepted! Correlation = 0.83 (vortex #22)
35 Processing detected swirling at (x, y) 36 56
Accepted! Correlation = 0.62 (vortex #23)
36 Processing detected swirling at (x, y) 58 57
Accepted! Correlation = 0.88 (vortex #24)
37 Processing detected swirling at (x, y) 2 58
Accepted! Correlation = 0.92 (vortex #25)
38 Processing detected swirling at (x, y) 25 58
Accepted! Correlation = 0.62 (vortex #26)
39 Processing detected swirling at (x, y) 47 58
---- Accepted vortices ----
27


In [99]:
# Plot Results
if xy_location != [0, 0]:
    x_location = int(xy_location[0])
    y_location = int(xy_location[1])
    detection_field_window = detection_field[y_location - 10:y_location + 10, x_location - 10:x_location + 10]
    x_index, y_index, u_data, v_data = fitting.window(vfield, x_location, y_location, 10)
    fitting.plot_quiver(x_index, y_index, u_data, v_data, detection_field_window)
if plot_method == 'detect':
    fitting.plot_detect(vortices_counterclockwise, vortices_clockwise, detection_field, flip_axis)
if plot_method == 'fields':
    fitting.plot_fields(vfield, vorticity)
if plot_method == 'fit':
    os.makedirs(output_directory, exist_ok=True)
    fitting.plot_accepted(vfield, vortices, detection_field, output_directory, time_step, output_format)
    fitting.plot_vortex(vfield, vortices, output_directory, time_step, output_format)
    output.write(vortices, output_directory, time_step)

r: 33299.239 gamma: -20694.17 xc: 191880.37 yc: 18532.42 correlation: 0.83 utheta: -0.06
r: 36149.116 gamma: -22280.15 xc: 477869.23 yc: 18532.42 correlation: 0.91 utheta: -0.06
r: 60375.720 gamma: 52818.47 xc: 88469.87 yc: 55597.46 correlation: 0.90 utheta: 0.09
r: 28875.664 gamma: 4543.45 xc: 262047.32 yc: 111194.93 correlation: 0.68 utheta: 0.02
r: 27545.903 gamma: 13135.91 xc: 116984.44 yc: 120461.24 correlation: 0.91 utheta: 0.05
r: 36309.675 gamma: -36304.29 xc: 448959.31 yc: 129727.35 correlation: 0.77 utheta: -0.10
r: 18776.156 gamma: 7471.70 xc: 218957.91 yc: 185324.81 correlation: 0.66 utheta: 0.04
r: 26722.248 gamma: -9152.39 xc: 19172.21 yc: 194591.13 correlation: 0.82 utheta: -0.03
r: 59254.269 gamma: 25725.76 xc: 295422.62 yc: 259454.89 correlation: 0.67 utheta: 0.04
r: 26248.771 gamma: -5314.59 xc: 126317.99 yc: 296519.72 correlation: 0.81 utheta: -0.02
r: 29931.115 gamma: -20400.55 xc: 343560.19 yc: 305786.06 correlation: 0.72 utheta: -0.07
r: 12450.679 gamma: 4067.03 x